# Lab 11-06 · The Continuous-Time Fourier Transform

Companion to [chapter 11-06](../../chapters/11-06-continuous-time-fourier-transform.md). Four parts: the FFT as an approximate Fourier transform, the convolution theorem, a sinc³ filter's bandwidth, and rise times in cascades. Run the cells in order.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.special import erf

## Part 1 — The FFT approximates the transform

Sample $x(t)$ every $\Delta t$ over a window long enough to hold it. Then $X(f_k) \approx \Delta t\sum_n x(n\Delta t)e^{-j2\pi f_kn\Delta t}$, which is $\Delta t$ times the FFT (with a phase correction for where the window starts).

In [2]:
def ft_by_fft(x, t0, dt, N):
    t = t0 + dt * np.arange(N)
    f = np.fft.fftfreq(N, dt)
    X = dt * np.fft.fft(x(t)) * np.exp(-2j * np.pi * f * t0)
    return np.fft.fftshift(f), np.fft.fftshift(X)

a = 1000.0
f, X = ft_by_fft(lambda t: np.where(t >= 0, np.exp(-a * t), 0.0), 0.0, 1e-6, 2 ** 16)
i = np.argmin(np.abs(f - a / (2 * np.pi)))              # the bin nearest a/2pi
print(f"exponential at f = {f[i]:.2f} Hz: FFT {abs(X[i]) * 1e3:.4f} ms, exact {abs(1 / (a + 2j * np.pi * f[i])) * 1e3:.4f} ms")

T = 1e-6
f, X = ft_by_fft(lambda t: ((t >= -T / 2) & (t < T / 2)).astype(float), -4e-6, 1e-9, 2 ** 15)
i = np.argmin(np.abs(f - 0.5e6))
print(f"1 us pulse at f = {f[i] / 1e6:.4f} MHz: FFT {abs(X[i]) * 1e6:.4f} us, exact {T * abs(np.sinc(f[i] * T)) * 1e6:.4f} us")

s = 1.0
f, X = ft_by_fft(lambda t: np.exp(-t ** 2 / (2 * s ** 2)), -40, 0.01, 2 ** 14)
i = np.argmin(np.abs(f - 0.2))
print(f"Gaussian at f = {f[i]:.4f}: FFT {abs(X[i]):.5f}, exact {s * np.sqrt(2 * np.pi) * np.exp(-2 * np.pi ** 2 * s ** 2 * f[i] ** 2):.5f}")

exponential at f = 152.59 Hz: FFT 0.7222 ms, exact 0.7218 ms
1 us pulse at f = 0.4883 MHz: FFT 0.6515 us, exact 0.6515 us
Gaussian at f = 0.2014: FFT 1.12541, exact 1.12541


## Part 2 — The convolution theorem, numerically

Convolve the 5 V, 2 ms pulse with the RC impulse response (τ = 1 ms) in time; then multiply their transforms and transform back. Compare the energies (Example 11-06.5).

In [3]:
dt = 1e-6
t = np.arange(0, 0.05, dt)
x = 5.0 * (t < 2e-3)
h = np.exp(-t / 1e-3) / 1e-3
y_time = np.convolve(x, h)[: len(t)] * dt
N = 2 * len(t)
y_freq = np.real(np.fft.ifft(np.fft.fft(x, N) * np.fft.fft(h, N)))[: len(t)] * dt
print("largest difference between time and frequency routes:", np.max(np.abs(y_time - y_freq)))
print(f"input energy {np.sum(x ** 2) * dt * 1e3:.2f} V^2 ms, output energy {np.sum(y_time ** 2) * dt * 1e3:.2f} V^2 ms")

largest difference between time and frequency routes: 6.217248937900877e-15
input energy 50.00 V^2 ms, output energy 28.41 V^2 ms


## Part 3 — The sinc³ filter's bandwidth

Compute the exact digital response $\lvert\sin(\pi fN/f_{MOD})/(N\sin(\pi f/f_{MOD}))\rvert^3$ for $N$ = 256 and $f_{MOD}$ = 128 kHz, and its continuous-time limit $\lvert\operatorname{sinc}(f/f_{DR})\rvert^3$.

In [4]:
N, fmod = 256, 128e3
fdr = fmod / N
H_exact = lambda f: abs(np.sin(np.pi * f * N / fmod) / (N * np.sin(np.pi * f / fmod))) ** 3
H_limit = lambda f: abs(np.sinc(f / fdr)) ** 3
f3_exact = brentq(lambda f: H_exact(f) - 2 ** -0.5, 1, 400)
f3_limit = brentq(lambda f: H_limit(f) - 2 ** -0.5, 1, 400)
print(f"-3 dB: exact {f3_exact:.2f} Hz ({f3_exact / fdr:.4f} f_DR), limit {f3_limit:.2f} Hz")
for fx in (60, 250, 499, 501):
    print(f"{fx:4d} Hz: {20 * np.log10(max(H_exact(fx), 1e-12)):8.2f} dB")

-3 dB: exact 130.97 Hz (0.2619 f_DR), limit 130.97 Hz
  60 Hz:    -0.62 dB
 250 Hz:   -11.77 dB
 499 Hz:  -161.89 dB
 501 Hz:  -161.99 dB


## Part 4 — Rise times in cascades (guided exercise)

**Predict first:** two isolated single-pole stages, each with a 1 ns rise time, are cascaded. Does the quadrature rule's 1.41 ns hold? Then run the cell, which computes rise times of cascades of $n$ identical single-pole stages and of Gaussian stages.

In [5]:
from scipy.special import gammainc, gammaincinv

def rise_time(step, t):
    return t[np.argmax(step >= 0.9)] - t[np.argmax(step >= 0.1)]

t = np.linspace(0, 20e-9, 400001)
dt = t[1] - t[0]
tau = 1e-9 / np.log(9)                        # one pole with a 1 ns rise time
h1 = np.exp(-t / tau) / tau
h = h1.copy()
for n in range(1, 5):
    if n > 1:
        h = np.convolve(h, h1)[: len(t)] * dt     # one more stage: convolve again
    s_conv = np.cumsum(h) * dt
    # The n-fold convolution of one-pole responses has a closed form, the Erlang
    # (gamma) distribution function, whose inverse gives the 10% and 90% times
    # exactly; the numerical convolution is a check on it.
    exact = (gammaincinv(n, 0.9) - gammaincinv(n, 0.1)) * tau   # 10% and 90% points
    print(f"{n} single-pole stages: rise {exact * 1e9:.4f} ns "
          f"(by numerical convolution {rise_time(s_conv, t) * 1e9:.3f}); quadrature rule {np.sqrt(n):.3f} ns")
sig = 1e-9 / 2.5631                           # Gaussian stage with a 1 ns rise time
for n in (1, 2):
    sg = 0.5 * (1 + erf((t - 10e-9) / (np.sqrt(2 * n) * sig)))
    print(f"{n} Gaussian stages: rise {rise_time(sg, t) * 1e9:.3f} ns")

1 single-pole stages: rise 1.0000 ns (by numerical convolution 1.000); quadrature rule 1.000 ns


2 single-pole stages: rise 1.5283 ns (by numerical convolution 1.528); quadrature rule 1.414 ns


3 single-pole stages: rise 1.9207 ns (by numerical convolution 1.920); quadrature rule 1.732 ns


4 single-pole stages: rise 2.2465 ns (by numerical convolution 2.245); quadrature rule 2.000 ns
1 Gaussian stages: rise 1.000 ns
2 Gaussian stages: rise 1.414 ns


**What to take away.** An FFT of finely sampled data, scaled by $\Delta t$, is the Fourier transform. Convolving in time and multiplying in frequency give the same answer. A data-sheet bandwidth can be derived from a filter's structure. And the quadrature rule for rise times is exact for Gaussian stages and only approximate for single poles, worsening as stages are added: the error grows toward about 17% for a long cascade, where the response tends to a Gaussian whose rise time is $2.5631/\ln 9 = 1.166$ times the quadrature rule's.